In [0]:
%python
# Extraemos los cometas

from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

cometas_data = {
    "fields": ["des","orbit_id","jd","cd","dist","dist_min","dist_max","v_rel","v_inf","t_sigma_f","h"],
    "data": [
        ["207P","106","2460374.511707941","2024-Mar-05 00:17","0.21981998656745","0.219819614761463","0.219820358373437","18.2745522286989","18.2738889343198","< 00:01",None],
        ["320P","30","2461188.162055015","2026-May-27 15:53","0.231497009844357","0.231493308105888","0.231500711587233","17.8522115916291","17.8515668543416","< 00:01",None],
        ["2021 N1","5","2461221.281247177","2026-Jun-29 18:45","0.16532603287701","0.165323645804498","0.165328419966898","16.6379689170075","16.6370002300635","< 00:01",None]
    ]
}

registros_cometas = []
for row in cometas_data["data"]:
    registro = dict(zip(cometas_data["fields"], row))
    registros_cometas.append({
        "designacion"        : registro["des"],
        "fecha_acercamiento" : registro["cd"],
        "distancia_au"       : float(registro["dist"]),
        "distancia_km"       : float(registro["dist"]) * 149597870,
        "velocidad_kms"      : float(registro["v_rel"]),
        "velocidad_kmh"      : float(registro["v_rel"]) * 3600,
        "magnitud_absoluta"  : float(registro["h"]) if registro["h"] else None,
        "tipo_astro"         : "Cometa",
        "ingestion_timestamp": datetime.now()
    })

schema = StructType([
    StructField("designacion",         StringType(),    True),
    StructField("fecha_acercamiento",  StringType(),    True),
    StructField("distancia_au",        DoubleType(),    True),
    StructField("distancia_km",        DoubleType(),    True),
    StructField("velocidad_kms",       DoubleType(),    True),
    StructField("velocidad_kmh",       DoubleType(),    True),
    StructField("magnitud_absoluta",   DoubleType(),    True),
    StructField("tipo_astro",          StringType(),    True),
    StructField("ingestion_timestamp", TimestampType(), True)
])

df_cometas = spark.createDataFrame(registros_cometas, schema=schema)
df_cometas.show()

+-----------+------------------+-----------------+--------------------+----------------+------------------+-----------------+----------+--------------------+
|designacion|fecha_acercamiento|     distancia_au|        distancia_km|   velocidad_kms|     velocidad_kmh|magnitud_absoluta|tipo_astro| ingestion_timestamp|
+-----------+------------------+-----------------+--------------------+----------------+------------------+-----------------+----------+--------------------+
|       207P| 2024-Mar-05 00:17| 0.21981998656745| 3.288460177391913E7|18.2745522286989| 65788.38802331605|             NULL|    Cometa|2026-07-19 01:43:...|
|       320P| 2026-May-27 15:53|0.231497009844357| 3.463145958408484E7|17.8522115916291| 64267.96172986476|             NULL|    Cometa|2026-07-19 01:43:...|
|    2021 N1| 2026-Jun-29 18:45| 0.16532603287701|2.4732422373950668E7|16.6379689170075|59896.688101226995|             NULL|    Cometa|2026-07-19 01:43:...|
+-----------+------------------+-----------------+--

In [0]:
%python
# Guardamos los cometas en bronze
df_cometas.createOrReplaceTempView("cometas_temp")

spark.sql("""
    CREATE TABLE IF NOT EXISTS nasa_dw.bronze.cometas_raw
    USING DELTA
    AS SELECT * FROM cometas_temp WHERE 1=0
""")

spark.sql("""
    MERGE INTO nasa_dw.bronze.cometas_raw AS tgt
    USING cometas_temp AS src
    ON tgt.designacion = src.designacion
    AND tgt.fecha_acercamiento = src.fecha_acercamiento
    WHEN NOT MATCHED THEN INSERT *
""")

print("✅ Cometas guardados en bronze.cometas_raw")

✅ Cometas guardados en bronze.cometas_raw


In [0]:
%sql
-- Validamos que los cometas se hayan guardado en bronze.cometas_raw
SELECT
    designacion,
    fecha_acercamiento,
    ROUND(distancia_km, 0)   AS distancia_km,
    ROUND(velocidad_kmh, 0)  AS velocidad_kmh,
    magnitud_absoluta,
    tipo_astro,
    ingestion_timestamp
FROM nasa_dw.bronze.cometas_raw
ORDER BY fecha_acercamiento DESC
LIMIT 10;

designacion,fecha_acercamiento,distancia_km,velocidad_kmh,magnitud_absoluta,tipo_astro,ingestion_timestamp
320P,2026-May-27 15:53,3.463146E7,64268.0,null,Cometa,2026-07-14T23:20:56.694Z
2021 N1,2026-Jun-29 18:45,2.4732422E7,59897.0,null,Cometa,2026-07-14T23:20:56.694Z
207P,2024-Mar-05 00:17,3.2884602E7,65788.0,null,Cometa,2026-07-14T23:20:56.694Z
